In [3]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os

# Clone your updated repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# Setup Kaggle-friendly paths
CAPTIONS_DIR = '/kaggle/working/captions'
os.makedirs(CAPTIONS_DIR, exist_ok=True)

def caption_path(input_dir):
    """Maps any hazy image dir to its captions.json under CAPTIONS_DIR."""
    key = input_dir.replace('/', '_').strip('_')
    out_dir = os.path.join(CAPTIONS_DIR, key)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, 'captions.json')

In [ ]:
# SateHaze1k Setup (Already mounted via Kaggle UI)
SATEHAZE_BASE = '/kaggle/input/datasets/xuxingxing233/satehaze1k'
SATEHAZE_LEVELS = ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick']

# RRSHID Download & Extraction
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
RRSHID_LEVELS = ['thin_fog', 'moderate_fog', 'thick_fog']

if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}
    print('RRSHID extracted.')

In [ ]:
# SateHaze1k Setup (Already mounted via Kaggle UI)
SATEHAZE_BASE = '/kaggle/input/datasets/xuxingxing233/satehaze1k'
SATEHAZE_LEVELS = ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick']

# RRSHID Download & Extraction
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
RRSHID_LEVELS = ['thin_fog', 'moderate_fog', 'thick_fog']

if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}
    print('RRSHID extracted.')

In [ ]:
import caption
from tqdm import tqdm

# Isolate execution to a single GPU to prevent accelerate deadlocks
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Initializing Qwen2-VL into VRAM...")
caption_fn, cleanup = caption.build_captioner(
    vlm_model_id="Qwen/Qwen2-VL-2B-Instruct",
    api_key="",
    api="local",
    device="cuda",
    max_side=512
)
print("Model loaded successfully.\n")

def process_directory(input_dir):
    """Processes a single directory using the pre-loaded VLM."""
    if not os.path.isdir(input_dir):
        return

    out_dir = os.path.dirname(caption_path(input_dir))
    os.makedirs(out_dir, exist_ok=True)
    cache_path = os.path.join(out_dir, "captions.json")
    
    cache = caption.load_existing(cache_path)
    images = caption.list_images(input_dir)
    todo = [f for f in images if f not in cache]
    
    if not todo:
        return
        
    print(f'Captioning: {input_dir}\nImages: {len(todo)}')

    for i, fname in enumerate(tqdm(todo, desc="Processing")):
        ipath = os.path.join(input_dir, fname)
        try:
            text = caption_fn(ipath)
            level = caption.parse_level(text)
            cache[fname] = {"caption": text, "level": level}
        except Exception as e:
            print(f"\n[WARN] Failed on {fname}: {e}")
            
        if (i + 1) % 20 == 0 or (i + 1) == len(todo):
            caption.save_cache(cache_path, cache)

# 1. Process SateHaze1k
for level in SATEHAZE_LEVELS:
    for split in ['train', 'test']:
        process_directory(f'{SATEHAZE_BASE}/{level}/{split}/hazy')

# 2. Process RRSHID
for level in RRSHID_LEVELS:
    for split in ['train', 'val', 'test']:
        process_directory(f'{RRSHID_BASE}/{level}/{split}/hazy')

cleanup()
print("\nAll datasets successfully captioned and saved to /kaggle/working/captions!")